# Titanic Survival — Classification

**Dataset:** Seaborn Titanic (891 rows)  
**Target:** `survived` (0 = No, 1 = Yes)  
**Models:** Logistic Regression, KNN, Naive Bayes, Decision Tree, SVM  
**Flow:** Load → Drop redundant cols → Impute → Encode → Split → Scale → Train → Evaluate

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

## 2. Load Data

In [ ]:
df = sns.load_dataset('titanic')
print(df.shape)
df.head(10)

## 3. Preprocessing

Drop redundant columns (`deck` >75% missing; `embark_town`, `alive`, `class`, `who`, `adult_male` are duplicates of other columns).

In [ ]:
drop_cols = ['deck', 'embark_town', 'alive', 'class', 'who', 'adult_male']
df.drop(columns=drop_cols, inplace=True)

df['age'].fillna(df['age'].mean(), inplace=True)
df.dropna(subset=['embarked'], inplace=True)

print('Shape after cleaning:', df.shape)
print('Remaining nulls:\n', df.isnull().sum())

In [ ]:
df.info()

In [ ]:
le = LabelEncoder()
df['sex']      = le.fit_transform(df['sex'])
df['embarked'] = le.fit_transform(df['embarked'])
df['alone']    = df['alone'].astype(int)
# Fill missing ages with the median age, then convert to standard int
df['age'] = df['age'].fillna(df['age'].median())
df = df.astype(int)
df.head()

## 4. Train-Test Split & Scaling

In [ ]:
X = df.drop('survived', axis=1)
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 5. Train & Compare Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'KNN (k=30)':          KNeighborsClassifier(n_neighbors=30),
    'Naive Bayes':         GaussianNB(),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'SVM (RBF)':           SVC(kernel='rbf')
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred   = model.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred)
    results.append({'Model': name, 'Accuracy': round(accuracy, 4)})

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
results_df

## 6. Best Model — Detailed Report

In [ ]:
best_name  = results_df.iloc[0]['Model']
best_model = models[best_name]
y_pred_best = best_model.predict(X_test_scaled)

print(f'Best Model: {best_name}\n')
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred_best))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_best, target_names=['Did Not Survive', 'Survived']))